# Bloque 3 — Demo Pipeline Completo

Este notebook recorre el pipeline de análisis de principio a fin usando los datos de **Carding Forums** (ya parseados en `01_carding_forums.ipynb`).

El objetivo es mostrar en vivo cada etapa: carga, normalización, análisis exploratorio, embeddings y visualización.

## Estructura
1. [Sección 1 — Load](#1-load)
2. [Sección 2 — Normalize](#2-normalize)
3. [Sección 3 — Analyze](#3-analyze)
4. [Sección 4 — Embed](#4-embed)
5. [Sección 5 — Visualize](#5-visualize)

## Setup — modelos Ollama necesarios

Ejecutar estos comandos **antes de empezar** si los modelos no están descargados.

```bash
ollama pull qwen3-embedding   # ~300MB — modelo de embeddings
ollama pull qwen2.5:14b        # ~9GB con Q4 — NER zero-shot
```

**⏸ En la demo se corta aquí — los modelos ya están descargados en la máquina del instructor.**

La sección de embeddings usa resultados precomputados para no bloquear la demo en Ollama.

In [ ]:
# Imports y configuración
import sys
from pathlib import Path

# Asegurar que la raíz del proyecto esté en sys.path
PROJECT_ROOT = Path().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx

from src.utils import load_forum, list_forums, merge_tables, load_or_compute, RESULTS_DIR, DATA_DIR
from src.timezone import build_user_timezone_profile

# Estilo de gráficos
plt.style.use('dark_background')
sns.set_palette('muted')

# Directorio de resultados para esta demo
DEMO_RESULTS = RESULTS_DIR / 'demo'
DEMO_RESULTS.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Datos: {DATA_DIR}")
print(f"Resultados: {DEMO_RESULTS}")

---

## 1. Load

Cargamos los 10 dumps de Carding Forums usando `load_forum()` con **auto-detect de formato y encoding**.

El parser maneja automáticamente:
- vBulletin SQL estándar y sus 5 variantes (prefijo `vb_`, columnas explícitas, encoding cp1251/UTF-8)
- Flat files delimitados por `:` (Cardingmafia.ws)

Cada foro devuelve un dict con tablas: `user`, `post`, `pmtext`, `userfield`.

In [ ]:
# Cargar todos los foros de Carding Forums
forum_paths = list_forums('Carding Forums')
print(f"Foros encontrados: {len(forum_paths)}")
for p in forum_paths:
    print(f"  {p.name}")

In [ ]:
# Cargar cada foro y mostrar estadísticas básicas
all_forums = []
for path in forum_paths:
    try:
        dfs = load_forum(path)
        all_forums.append(dfs)
        u = len(dfs.get('user', []))
        p = len(dfs.get('post', []))
        print(f"  ✓ {path.stem}: {u:,} usuarios, {p:,} posts")
    except Exception as e:
        print(f"  ✗ {path.stem}: {e}")

print(f"\nForos cargados exitosamente: {len(all_forums)}")

In [ ]:
# Merge en DataFrames unificados con columna 'forum'
all_users = merge_tables(all_forums, 'user')
all_posts = merge_tables(all_forums, 'post')

print(f"Total usuarios: {len(all_users):,}")
print(f"Total posts:    {len(all_posts):,}")
print(f"Foros únicos:   {all_users['forum'].nunique()}")
print(f"\nColumnas users: {list(all_users.columns)}")
print(f"Columnas posts: {list(all_posts.columns)}")

---

## 2. Normalize

Antes de analizar, limpiamos los datos:
- **Deduplicación**: eliminar usuarios registrados en múltiples foros con el mismo `userid` si tienen el mismo email/hash
- **Fechas a UTC**: normalizar `dateline` a datetime aware UTC
- **Texto limpio**: eliminar NULLs explícitos, strings vacíos

In [ ]:
# Normalizar fechas de posts a UTC datetime
def normalize_dateline(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte la columna dateline (Unix timestamp) a datetime UTC."""
    df = df.copy()
    if 'dateline' in df.columns:
        df['dateline'] = pd.to_datetime(
            pd.to_numeric(df['dateline'], errors='coerce'),
            unit='s',
            utc=True,
            errors='coerce'
        )
    return df

all_posts = normalize_dateline(all_posts)

# Verificar rango temporal
valid_dates = all_posts['dateline'].dropna()
print(f"Posts con fecha válida: {len(valid_dates):,} / {len(all_posts):,}")
if len(valid_dates) > 0:
    print(f"Rango temporal: {valid_dates.min()} → {valid_dates.max()}")

In [ ]:
# Deduplicar usuarios: misma combinación forum+userid
n_before = len(all_users)
all_users = all_users.drop_duplicates(subset=['forum', 'userid'])
n_after = len(all_users)
print(f"Deduplicación: {n_before:,} → {n_after:,} usuarios ({n_before - n_after:,} duplicados eliminados)")

---

## 3. Analyze

Análisis exploratorio en tres dimensiones: estadística descriptiva, red de interacciones y análisis temporal.

In [ ]:
# Estadística descriptiva por foro
user_stats = all_users.groupby('forum').agg(
    usuarios=('userid', 'count'),
).reset_index()

post_stats = all_posts.groupby('forum').agg(
    posts=('userid', 'count'),
).reset_index()

stats = user_stats.merge(post_stats, on='forum', how='left').fillna(0)
stats['posts_por_usuario'] = (stats['posts'] / stats['usuarios']).round(1)
stats = stats.sort_values('usuarios', ascending=False)
print(stats.to_string(index=False))

In [ ]:
# Power-law: distribución de posts por usuario
if 'userid' in all_posts.columns:
    post_counts = all_posts.groupby(['forum', 'userid']).size().reset_index(name='n_posts')
    
    fig, ax = plt.subplots(figsize=(10, 4))
    # Histograma log-log
    counts_values = post_counts['n_posts'].values
    bins = np.logspace(0, np.log10(counts_values.max() + 1), 50)
    ax.hist(counts_values, bins=bins, color='#E94E4E', alpha=0.8, edgecolor='none')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Posts por usuario')
    ax.set_ylabel('Número de usuarios')
    ax.set_title('Distribución power-law de actividad (todos los foros)')
    plt.tight_layout()
    plt.show()
    print(f"Top 1% de usuarios genera: {post_counts.nlargest(int(len(post_counts)*0.01), 'n_posts')['n_posts'].sum() / post_counts['n_posts'].sum():.1%} de los posts")

In [ ]:
# Grafo de interacciones (muestra: foro más activo)
top_forum = stats.iloc[0]['forum'] if len(stats) > 0 else None

if top_forum and 'parentid' in all_posts.columns:
    forum_posts = all_posts[all_posts['forum'] == top_forum].copy()
    forum_posts['parentid'] = pd.to_numeric(forum_posts.get('parentid', 0), errors='coerce').fillna(0).astype(int)
    
    edges = forum_posts[forum_posts['parentid'] > 0][['parentid', 'userid']].dropna()
    edges = edges.head(50000)  # limitar para demo
    
    G = nx.from_pandas_edgelist(
        edges,
        source='parentid',
        target='userid',
        create_using=nx.DiGraph()
    )
    print(f"Grafo de {top_forum}: {G.number_of_nodes():,} nodos, {G.number_of_edges():,} aristas")
    
    # Top 10 por betweenness (muestra)
    btw = nx.betweenness_centrality(G, k=min(500, G.number_of_nodes()), normalized=True)
    top_btw = sorted(btw.items(), key=lambda x: x[1], reverse=True)[:10]
    print(f"\nTop 10 usuarios por betweenness centrality:")
    for uid, score in top_btw:
        username = all_users[(all_users['forum'] == top_forum) & (all_users['userid'] == uid)]['username'].values
        name = username[0] if len(username) > 0 else str(uid)
        print(f"  {name}: {score:.4f}")

In [ ]:
# Análisis temporal: evolución mensual de registros
if 'joindate' in all_users.columns:
    users_time = all_users.copy()
    users_time['joindate'] = pd.to_datetime(
        pd.to_numeric(users_time['joindate'], errors='coerce'),
        unit='s',
        utc=True,
        errors='coerce'
    )
    users_time = users_time.dropna(subset=['joindate'])
    users_time = users_time[(users_time['joindate'].dt.year >= 2005) & (users_time['joindate'].dt.year <= 2024)]
    
    monthly = users_time.groupby(users_time['joindate'].dt.to_period('M')).size()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(monthly.index.astype(str), monthly.values, color='#E94E4E', linewidth=1.5)
    ax.fill_between(range(len(monthly)), monthly.values, alpha=0.3, color='#E94E4E')
    ax.set_xticks(range(0, len(monthly), max(1, len(monthly)//10)))
    ax.set_xticklabels([str(monthly.index[i]) for i in range(0, len(monthly), max(1, len(monthly)//10))], rotation=45)
    ax.set_title('Evolución temporal de registros — Carding Forums')
    ax.set_ylabel('Nuevos usuarios por mes')
    plt.tight_layout()
    plt.show()

In [ ]:
# Análisis de timezone
if 'dateline' in all_posts.columns and all_posts['dateline'].notna().any():
    tz_profile = build_user_timezone_profile(all_posts)
    
    offset_dist = tz_profile['inferred_utc_offset'].value_counts().sort_index()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(offset_dist.index, offset_dist.values, color='#E94E4E', alpha=0.8)
    ax.set_xlabel('UTC offset inferido')
    ax.set_ylabel('Número de usuarios')
    ax.set_title('Distribución de timezone inferida — Carding Forums')
    ax.axvline(3, color='#FFD700', linestyle='--', label='UTC+3 (Rusia/Ucrania)')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    top_offset = offset_dist.idxmax()
    print(f"Offset más frecuente: UTC{top_offset:+d}")

---

## 4. Embed

Generamos embeddings de usuario usando `qwen3-embedding` vía Ollama.

Los resultados se cachean en `results/demo/` con `load_or_compute()` para que la demo no bloquee en Ollama.

El código de generación está aquí para mostrarlo; la ejecución real sobre el dataset completo se hizo previamente.

In [ ]:
# Código de generación de embeddings (referencia — se muestra pero no se ejecuta completo en demo)
# Para regenerar desde cero, descomentar y ejecutar con Ollama activo:

# from src.embeddings import embed_users
# def compute_embeddings():
#     user_ids, vectors = embed_users(all_posts, min_posts=3)
#     return {'user_ids': np.array(user_ids), 'vectors': vectors}
#
# result = load_or_compute(
#     DEMO_RESULTS / 'demo_embeddings.npz',
#     compute_embeddings
# )

print("El código anterior genera y cachea los embeddings de todos los usuarios.")
print("En la demo se carga el archivo precomputado si existe.")

In [ ]:
# Demo en vivo: embedding de una muestra pequeña de usuarios (100)
# Ejecutar solo si Ollama está disponible con qwen3-embedding

DEMO_EMBED_PATH = DEMO_RESULTS / 'demo_embeddings.npz'

if DEMO_EMBED_PATH.exists():
    # Cargar precomputado
    cached = np.load(DEMO_EMBED_PATH, allow_pickle=False)
    demo_user_ids = cached['user_ids'].tolist()
    demo_vectors = cached['vectors']
    print(f"Embeddings cargados desde caché: {len(demo_user_ids):,} usuarios, dim={demo_vectors.shape[1]}")
else:
    print("No hay embeddings precomputados.")
    print("Para generar: activar Ollama con qwen3-embedding y descomentar la celda anterior.")
    
    # Muestra pequeña en vivo (100 usuarios con más posts)
    try:
        import ollama as _ollama_test
        _ollama_test.list()  # verificar que Ollama responde
        
        from src.embeddings import embed_users
        sample_posts = all_posts.groupby('userid').filter(lambda x: len(x) >= 3).copy()
        top_users = sample_posts.groupby('userid').size().nlargest(100).index
        sample_posts = sample_posts[sample_posts['userid'].isin(top_users)]
        
        print(f"Generando embeddings para muestra de {sample_posts['userid'].nunique()} usuarios...")
        demo_user_ids, demo_vectors = embed_users(sample_posts, min_posts=3)
        print(f"Embeddings generados: {len(demo_user_ids)} usuarios, dim={demo_vectors.shape[1]}")
    except Exception as e:
        print(f"Ollama no disponible: {e}")
        demo_user_ids = []
        demo_vectors = np.empty((0, 4096), dtype=np.float32)

---

## 5. Visualize

Visualizamos los resultados: grafo de interacciones, heatmap temporal y scatter de clusters de embeddings.

In [ ]:
# Heatmap temporal: actividad por hora y día de la semana
if 'dateline' in all_posts.columns and all_posts['dateline'].notna().any():
    posts_time = all_posts.dropna(subset=['dateline']).copy()
    posts_time['hour'] = posts_time['dateline'].dt.hour
    posts_time['weekday'] = posts_time['dateline'].dt.weekday
    
    heatmap_data = posts_time.groupby(['weekday', 'hour']).size().unstack(fill_value=0)
    
    day_names = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
    heatmap_data.index = [day_names[i] for i in heatmap_data.index]
    
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(heatmap_data, cmap='YlOrRd', ax=ax, linewidths=0.1, cbar_kws={'label': 'Posts'})
    ax.set_title('Actividad de posts por hora UTC y día de la semana — Carding Forums')
    ax.set_xlabel('Hora UTC')
    plt.tight_layout()
    plt.show()

In [ ]:
# Scatter de clusters UMAP de embeddings
if 'demo_vectors' in dir() and len(demo_vectors) > 10:
    try:
        import umap
        import plotly.express as px
        
        reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=min(15, len(demo_vectors)-1))
        coords_2d = reducer.fit_transform(demo_vectors)
        
        # Obtener usernames para hover
        uid_to_name = dict(zip(all_users['userid'], all_users['username']))
        usernames = [uid_to_name.get(uid, str(uid)) for uid in demo_user_ids]
        forums_per_user = [
            all_users[all_users['userid'] == uid]['forum'].values[0]
            if len(all_users[all_users['userid'] == uid]) > 0 else 'unknown'
            for uid in demo_user_ids
        ]
        
        df_plot = pd.DataFrame({
            'x': coords_2d[:, 0],
            'y': coords_2d[:, 1],
            'username': usernames,
            'forum': forums_per_user,
        })
        
        fig = px.scatter(
            df_plot, x='x', y='y',
            color='forum',
            hover_data=['username'],
            title='Clusters de estilo de escritura (UMAP de embeddings)',
            template='plotly_dark',
            width=900, height=600,
        )
        fig.show()
    except Exception as e:
        print(f"UMAP/Plotly no disponible o embeddings vacíos: {e}")
else:
    print("Sin embeddings cargados — visualización UMAP no disponible en esta ejecución.")

In [ ]:
# Resumen final del pipeline
print("=" * 50)
print("RESUMEN DEL PIPELINE")
print("=" * 50)
print(f"Foros procesados:     {len(all_forums)}")
print(f"Usuarios totales:     {len(all_users):,}")
print(f"Posts totales:        {len(all_posts):,}")
if 'demo_vectors' in dir() and len(demo_vectors) > 0:
    print(f"Embeddings generados: {len(demo_vectors):,} usuarios")
print("=" * 50)